# 02 - Certificate collection (TLS probe + Certificate Transparency)

**How to use this notebook: Runtime -> Run all. If the session dies at any
point, open it again and Run all again.** Every step checkpoints to disk and
resumes; nothing is ever collected twice.

Two collections run here, in order:

* **Part A - TLS probe.** Connects to each domain in the probe universe and
  retrieves its live certificate. Most DGA domains will fail with NXDOMAIN -
  they were generated but never registered. That is a finding, not a fault.
* **Part B - Certificate Transparency.** For domains that DID return a
  certificate, fetches their public issuance history from crt.sh (first-seen
  date, certificates ever issued). Slow and rate-limited; runs only after
  Part A has something to look up.

Ledgers live on local disk, not Drive - Drive's FUSE mount lacks the POSIX
locking SQLite needs - and are backed up to Drive every few thousand records,
so even a full runtime loss resumes from the backup.

In [ ]:
# --- standard header ---
from google.colab import drive
drive.mount('/content/drive')

import os, sys, subprocess, getpass
REPO = '/content/secure-dns-trust-ai'
URL  = 'github.com/sandesh20lamichhane/secure-dns-trust-ai.git'
if os.path.isdir(REPO):
    subprocess.run(['git','-C',REPO,'pull','-q'], check=False)
else:
    TOKEN = getpass.getpass('GitHub PAT: ')
    subprocess.run(['git','clone','-q',f'https://{TOKEN}@{URL}',REPO], check=True)
sys.path.insert(0, REPO)
os.environ['DNSTRUST_CONFIG_DIR'] = f'{REPO}/configs'

from src.utils import config, manifest, seeds
P = config.paths(); config.ensure_tree(P); seeds.set_all(42)
print('repo', manifest.git_sha(REPO))

In [ ]:
!pip -q install dnspython cryptography pyarrow zstandard

In [ ]:
# one-time cleanup: an old bug created the ledger path as a directory
import shutil, os
for bad in [P['local']['ledger']]:
    if os.path.isdir(bad):
        shutil.rmtree(bad); print('removed stray directory:', bad)
print('paths clean')

## Part A - TLS probe

Runs to completion in one cell (about 2-4 hours for 50k domains), writing a
shard and a ledger backup after every batch of 2,000. Interrupting loses at
most one batch; re-running resumes.

In [ ]:
import pandas as pd
from src.collect.ledger import Ledger
from src.collect import tls_prober
from src.utils.logging_setup import get_logger

log = get_logger('collect_tls', P['artifacts']['logs'])
ledger = Ledger(P['local']['ledger'],
                drive_backup=f"{P['artifacts']['logs']}/certificate_ledger_backup.db")
print('restored from Drive backup:', ledger.restore_from_backup())

universe = pd.read_parquet(f"{P['data']['interim']}/probe_universe.parquet")
ledger.enqueue(universe[['domain','source','label']]
               .assign(label=universe['label'].astype(str)).to_dict('records'))
print(ledger.summary())

In [ ]:
# Runs until nothing is pending. Progress prints after every batch.
summary = tls_prober.run_collection(
    ledger,
    out_dir=f"{P['data']['collected']}/tls_probe",
    batch_size=2000,
    concurrency=100,
    max_batches=None,        # to completion; safe to interrupt and re-run
    logger=log)
ledger.retire_exhausted()
ledger.backup()
for k, v in sorted(ledger.summary().items()):
    print(f'{k:20s} {v}')

### Coverage report

The number the paper needs: certificate-retrieval rate by class and source.
Expect the DGA rows dominated by NXDOMAIN and the Tranco / URLhaus rows
carrying the certificates - that asymmetry is the empirical justification for
fusing a lexical branch with a certificate branch.

In [ ]:
from src.utils.io import read_shards

tls = read_shards(f"{P['data']['collected']}/tls_probe", 'tls_probe')
print('certificates collected:', len(tls))

if len(tls):
    got = set(tls['domain'])
    universe['has_cert'] = universe['domain'].isin(got)
    cov = universe.groupby(['label','source'])['has_cert'].agg(n='count', with_cert='sum')
    cov['rate'] = (cov['with_cert']/cov['n']).round(4)
    display(cov)

    q = pd.read_sql('SELECT status, label, COUNT(*) n FROM domains GROUP BY status, label',
                    ledger.conn)
    display(q.pivot(index='status', columns='label', values='n').fillna(0).astype(int))

## Part B - Certificate Transparency

Only domains that returned a certificate are looked up; CT queries on
never-registered DGA domains return nothing and waste rate-limited requests.
Anything fetched by any earlier attempt is detected from the shards on Drive
and never repeated.

Roughly four to six seconds per domain at eight in parallel - about one to two
hours for ~6,000 domains. Shards flush every 250 rows, so an abrupt kill loses
almost nothing. If crt.sh starts erroring in bulk, it is rate-limiting: stop,
wait an hour, Run all again.

In [ ]:
from src.collect import ct_collect

ct_log = get_logger('collect_ct', P['artifacts']['logs'])
ct_ledger = ct_collect.open_ledger(P)
print('restored from Drive backup:', ct_ledger.restore_from_backup())

live = sorted(set(tls['domain'])) if len(tls) else []
ct_ledger.enqueue([{'domain': d, 'source': 'tls_live'} for d in live])

ct_old = read_shards(f"{P['data']['collected']}/crtsh", 'crtsh')
already = set(ct_old['domain']) if len(ct_old) else set()
for d in already:
    ct_ledger.mark(d, 'success')
ct_ledger.commit()

print('to look up:', len(live), '| already done:', len(already),
      '| pending:', len(ct_ledger.pending()))

In [ ]:
summary = ct_collect.run(
    ct_ledger,
    out_dir=f"{P['data']['collected']}/crtsh",
    batch_size=400,
    concurrency=8,
    max_batches=None,        # to completion; safe to interrupt and re-run
    logger=ct_log)
ct_ledger.retire_exhausted()
ct_ledger.backup()
for k, v in sorted(ct_ledger.summary().items()):
    print(f'{k:20s} {v}')

In [ ]:
ct = read_shards(f"{P['data']['collected']}/crtsh", 'crtsh')
print('CT rows:', len(ct))
if len(ct) and len(tls):
    print(f'coverage of certificate-bearing domains: '
          f'{100*ct["domain"].nunique()/tls["domain"].nunique():.1f}%')
    print('domains with zero CT history:', int((ct["ct_log_count"]==0).sum()))
    display(ct[['domain','ct_log_count','ct_first_seen',
                'days_since_first_ct_seen','ct_distinct_issuers']].head(8))

---

Both parts done when both summaries show nothing pending. CT is enrichment: if
it stalls on rate limits for days, record the coverage percentage achieved and
move on - the certificate branch already works on the collected certificates.

**Next:** `04_split_creation`, then `03_feature_engineering` (in that order -
the n-gram model fits on training benign domains only).